<a href="https://colab.research.google.com/github/RohitKumar2705/LLM/blob/main/GPT_TO_llama2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "sentencepiece",    # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

huggingface_hub version: 1.5.0
sentencepiece version: 0.2.1
torch version: 2.10.0+cpu


In [3]:
import torch
import torch.nn as nn
class RMSnorm(nn.Module):
  def __init__(self,emb_dim,eps = 1e-5):

    super().__init__()
    self.eps = eps
    self.emb_dim = emb_dim
    self.weight = nn.Parameter(torch.ones(emb_dim)).float()

  def forward(self,x):
    means = x.pow(2).mean(dim = -1, keepdim = True)
    x_normed = x*torch.rsqrt(means +self.eps)
    return(x_normed + self.weight).to(dtype = x.type)




In [4]:
import torch
import torch.nn as nn
class RMSnorm(nn.Module):
  def __init__(self,emb_dim,eps = 1e-5):

    super().__init__()
    self.eps = eps
    self.emb_dim = emb_dim
    self.weight = nn.Parameter(torch.ones(emb_dim)).float()

  def forward(self,x):
    means = x.pow(2).mean(dim = -1, keepdim = True)
    x_normed = x*torch.rsqrt(means +self.eps)
    return(x_normed * self.weight).to(dtype = x.dtype) # Changed + to *

torch.manual_seed(123)
example_batch = torch.randn(2,3,4)
rms_norm = RMSnorm(emb_dim = example_batch.shape[-1])
rmsnorm_pytorch = torch.nn.RMSNorm(example_batch.shape[-1], eps=1e-5)

assert torch.allclose(rms_norm(example_batch), rmsnorm_pytorch(example_batch))

In [5]:
class SiLU(nn.Module):
  def __init__(self,x):
    super(SiLU,self).__init__()

  def forward(self,x):
    return x*torch.sigmoid(x)
# llama use silu fumnction instead of gelu

In [6]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
    self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
    self.fc3 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
    self.silu = SiLU()

  def forward(self,x):
    x_fc1 = self.fc1(x)
    x_fc2 = self.fc2(x)
    x = self.silu(x_fc1)*x_fc2
    return self.fc3(x)

In [7]:
def precompute_rope_params(head_dim,theta_base = 10_000,context_length = 4096):
  assert head_dim % 2 == 0, "Embedding dimension must be even"
  inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
   # Generate position indices
  positions = torch.arange(context_length)

  # Compute the angles
  angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)

  # expand angle to match
  angles = torch.cat([angles,angles],dim = 1)

  # precompute sin and cosine
  cos = torch.cos(angles)
  sin = torch.sin(angles)
  return cos,sin

def compute_rope(x,sin,cos):
  batch_size,num_heads,seq_len,head_dim = x.shape

  assert head_dim % 2 == 0, "Head dimension must be even"

  # Split x into first half and second half
  x1 = x[..., : head_dim // 2]  # First half
  x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
  cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
  sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
  rotated = torch.cat((-x2, x1), dim=-1) # Uncommented this line
  x_rotated = (x * cos) + (rotated * sin)

  return x_rotated.to(dtype=x.dtype)

# settings
batch_size = 2
context_len = 5
num_heads = 4
head_dim = 16

cos,sin = precompute_rope_params(head_dim=head_dim, context_length=context_len)

torch.manual_seed(123)
queries = torch.randn(batch_size, num_heads, context_len, head_dim)
keys = torch.randn(batch_size, num_heads, context_len, head_dim)


# Apply rotary position embeddings
queries_rot = compute_rope(queries, cos, sin)
keys_rot = compute_rope(keys, cos, sin)


In [8]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,num_heads,head_dim = None):
    super().__init__()
    assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out//num_heads
    self.W_query = nn.Linear(d_in, d_out, bias=False)
    self.W_key = nn.Linear(d_in, d_out, bias=False)
    self.W_value = nn.Linear(d_in, d_out, bias=False)
    self.out_proj = nn.Linear(d_out, d_out, bias=False)  # Linear layer to combine head outputs
         # self.dropout = nn.Dropout(dropout)
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    cos, sin = precompute_rope_params(head_dim=self.head_dim, context_length=context_length)
    self.register_buffer("cos", cos)
    self.register_buffer("sin", sin)


def forward(self,x):
  b,num_tokens,d_in = x.shape
  keys = self.W_key(x)
  queries = self.W_query(x)
  values = self.W_value(x)
  keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
  values = values.view(b, num_tokens, self.num_heads, self.head_dim)
  queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

  # transpose
  keys = keys.transpose(1,2)
  values = values.transpose(1,2)
  queries = queries.transpose(1,2)

  keys = compute_rope(keys,self.cos,self.sin)
  queries = compute_rope(queries,self.cos,self.sin)

  attn_scores = queries@keys.transpose(2,3)
  mask_bool = self.mask_bool()[:num_tokens,:num_tokens]

  # mask to fill attention

  attn_scores.masked_fill_(mask_bool,-float.inf)
  attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
  context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
  context_vec = context_vec.reshape(b, num_tokens, self.d_out)
  context_vec = self.out_proj(context_vec)  # optional projection

  return context_vec





In [9]:
import torch
import torch.nn as nn

# Re-defining MultiHeadAttention with corrected indentation and mask usage
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,num_heads,head_dim = None):
    super().__init__()
    assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out//num_heads
    self.W_query = nn.Linear(d_in, d_out, bias=False)
    self.W_key = nn.Linear(d_in, d_out, bias=False)
    self.W_value = nn.Linear(d_in, d_out, bias=False)
    self.out_proj = nn.Linear(d_out, d_out, bias=False)  # Linear layer to combine head outputs
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    cos, sin = precompute_rope_params(head_dim=self.head_dim, context_length=context_length)
    self.register_buffer("cos", cos)
    self.register_buffer("sin", sin)

  # Corrected indentation for forward method
  def forward(self,x):
    b,num_tokens,d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    # transpose
    keys = keys.transpose(1,2)
    values = values.transpose(1,2)
    queries = queries.transpose(1,2)

    keys = compute_rope(keys,self.cos,self.sin)
    queries = compute_rope(queries,self.cos,self.sin)

    attn_scores = queries@keys.transpose(2,3)
    # Corrected: self.mask_bool() to self.mask and cast to bool
    mask_bool = self.mask[:num_tokens,:num_tokens].bool()

    # mask to fill attention
    attn_scores.masked_fill_(mask_bool,float('-inf')) # Changed -float.inf to float('-inf')
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    # Shape: (b, num_tokens, num_heads, head_dim)
    context_vec = (attn_weights @ values).transpose(1, 2)

    # Combine heads, where self.d_out = self.num_heads * self.head_dim
    context_vec = context_vec.reshape(b, num_tokens, self.d_out)
    context_vec = self.out_proj(context_vec)  # optional projection

    return context_vec

batch_size = 1
context_len = 100
max_context_len = 4096
embed_dim = 128
num_heads = 4


example_batch = torch.randn((batch_size, context_len, embed_dim))

mha = MultiHeadAttention(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=max_context_len,
    num_heads=num_heads
)

mha(example_batch)

del mha

In [10]:
class transformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att = MultiHeadAttention(d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dtype=cfg["dtype"])
    self.ff = FeedForward(cfg)

    self.norm1 = RMSnorm(cfg["emb_dim"])
    self.norm2 = RMSnorm(cfg["emb_dim"])
    self.norm3 = RMSnorm(cfg["emb_dim"])
  def forward(self,x):
    shortcut = x
    x = self.norm1(x)
    x = self.att(x)
    x = x +shortcut

    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = x +shortcut
    return x


In [11]:
# class GPTMODULE
class Llama2Model(nn.Module):
  def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])


        self.trf_blocks = nn.Sequential(
            *[transformerBlock(cfg) for _ in range(cfg["n_layers"])])


        # self.final_norm = LayerNorm(cfg["emb_dim"])
        self.final_norm = RMSNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

  def forward(self, in_idx):
        # batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        x = tok_embeds
        # x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [12]:
class Llama2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])


        self.trf_blocks = nn.Sequential(
            *[transformerBlock(cfg) for _ in range(cfg["n_layers"])])


        # self.final_norm = LayerNorm(cfg["emb_dim"])
        self.final_norm = RMSNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

    def forward(self, in_idx):
        # batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        x = tok_embeds
        # x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [13]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

In [14]:
GPT_CONFIG_1558M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 1600,         # Embedding dimension
    "n_heads": 25,           # Number of attention heads
    "n_layers": 48,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

In [15]:
LLAMA2_CONFIG_7B = {
    "vocab_size": 32000,     # Vocabulary size
    "context_length": 4096,  # Context length
    "emb_dim": 4096,         # Embedding dimension
    "n_heads": 32,           # Number of attention heads
    "n_layers": 32,          # Number of layers
    "hidden_dim": 11008,     # NEW: Size of the intermediate dimension in FeedForward
    "dtype": torch.bfloat16  # NEW: Lower-precision dtype to reduce memory usage
}

In [ ]:
import torch
import torch.nn as nn

# Redefine RMSnorm from cell H04YwOU8Eebq with previous fix
class RMSnorm(nn.Module):
  def __init__(self,emb_dim,eps = 1e-5):
    super().__init__()
    self.eps = eps
    self.emb_dim = emb_dim
    self.weight = nn.Parameter(torch.ones(emb_dim)).float()

  def forward(self,x):
    means = x.pow(2).mean(dim = -1, keepdim = True)
    x_normed = x*torch.rsqrt(means +self.eps)
    return(x_normed * self.weight).to(dtype = x.dtype)

# Redefine SiLU from cell y2L0_jdLFAju
class SiLU(nn.Module):
  def __init__(self):
    super(SiLU,self).__init__()

  def forward(self,x):
    return x*torch.sigmoid(x)

# Redefine FeedForward from cell QbrLNY9FGKeI
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
    self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
    self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False) # Changed from cfg["emb_dim"] to cfg["hidden_dim"] and output to cfg["emb_dim"]
    self.silu = SiLU()

  def forward(self,x):
    x_fc1 = self.fc1(x)
    x_fc2 = self.fc2(x)
    x = self.silu(x_fc1)*x_fc2
    return self.fc3(x)

# Redefine precompute_rope_params and compute_rope from cell RgfVMH1lIvau with previous fix
def precompute_rope_params(head_dim,theta_base = 10_000,context_length = 4096):
  assert head_dim % 2 == 0, "Embedding dimension must be even"
  inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
  positions = torch.arange(context_length)
  angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
  angles = torch.cat([angles,angles],dim = 1)
  cos = torch.cos(angles)
  sin = torch.sin(angles)
  return cos,sin

def compute_rope(x,sin,cos):
  batch_size,num_heads,seq_len,head_dim = x.shape
  assert head_dim % 2 == 0, "Head dimension must be even"
  x1 = x[..., : head_dim // 2]
  x2 = x[..., head_dim // 2 :]
  cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
  sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)
  rotated = torch.cat((-x2, x1), dim=-1)
  x_rotated = (x * cos) + (rotated * sin)
  return x_rotated.to(dtype=x.dtype)

# Redefine MultiHeadAttention from cell TkFnmrFgQ_3w with `dtype` argument and previous fixes
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,num_heads,head_dim = None, dtype=None):
    super().__init__()
    assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out//num_heads
    self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
    self.W_key = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
    self.W_value = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
    self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    cos, sin = precompute_rope_params(head_dim=self.head_dim, context_length=context_length)
    self.register_buffer("cos", cos)
    self.register_buffer("sin", sin)

  def forward(self,x):
    b,num_tokens,d_in = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    keys = keys.transpose(1,2)
    values = values.transpose(1,2)
    queries = queries.transpose(1,2)

    keys = compute_rope(keys,self.cos,self.sin)
    queries = compute_rope(queries,self.cos,self.sin)

    attn_scores = queries@keys.transpose(2,3)
    mask_bool = self.mask[:num_tokens,:num_tokens].bool()

    attn_scores.masked_fill_(mask_bool,float('-inf'))
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    context_vec = (attn_weights @ values).transpose(1, 2)

    context_vec = context_vec.reshape(b, num_tokens, self.d_out)
    context_vec = self.out_proj(context_vec)

    return context_vec

# Redefine transformerBlock from cell nnVsIZbwR3wX
class transformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att = MultiHeadAttention(d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dtype=cfg["dtype"]) # Now MultiHeadAttention accepts dtype
    self.ff = FeedForward(cfg)

    self.norm1 = RMSnorm(cfg["emb_dim"])
    self.norm2 = RMSnorm(cfg["emb_dim"])
    self.norm3 = RMSnorm(cfg["emb_dim"])
  def forward(self,x):
    shortcut = x
    x = self.norm1(x)
    x = self.att(x)
    x = x +shortcut

    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = x +shortcut
    return x

# Redefine Llama2Model from cell TMP39K9fTsur, using RMSnorm instead of RMSNorm
class Llama2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.Sequential(
            *[transformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = RMSnorm(cfg["emb_dim"]) # Changed RMSNorm to RMSnorm

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

    def forward(self, in_idx):
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# Original line from KngRRibST_Py
model = Llama2Model(LLAMA2_CONFIG_7B)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

In [ ]:
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

In [ ]:
from huggingface_hub import hf_hub_download

tokenizer_file = hf_hub_download(
    repo_id="meta-llama/Llama-2-7b",
    filename="tokenizer.model",
    local_dir="Llama-2-7b"
)

In [ ]:
import sentencepiece as spm


class LlamaTokenizer:
    def __init__(self, tokenizer_file):
        sp = spm.SentencePieceProcessor()
        sp.load(tokenizer_file)
        self.tokenizer = sp

    def encode(self, text):
        return self.tokenizer.encode(text, out_type=int)

    def decode(self, ids):
        return self.tokenizer.decode(ids)


tokenizer = LlamaTokenizer(tokenizer_file)

In [ ]:
from previous_chapters import generate, text_to_token_ids, token_ids_to_text
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=LLAMA2_CONFIG_7B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
weights_file = hf_hub_download(
   repo_id="meta-llama/Llama-2-7b",
   filename="consolidated.00.pth",
   local_dir="Llama-2-7b"
)

In [ ]:
weights = torch.load(weights_file, weights_only=True)

In [ ]:
list(weights.keys())[:15]